In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import altair as alt
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.stats import spearmanr
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

# Enable Altair's data transformer for large datasets
alt.data_transformers.enable('default', max_rows=None)
alt.renderers.enable('default')

# Standardize the features using RobustScaler
robust_scaler = RobustScaler()

In [2]:
# Load S&P 500 Data (monthly)
sp500_data = pd.read_csv('../../data/yfinance/sp500_yfinance_monthly.csv')
# sp500_data = pd.read_csv('../../data/final/sp500_monthly_data.csv')
sp500_data_edgar = pd.read_csv('../../data/edgar_sec/sp500_edgar_sec.csv')
# all columns capitalized
sp500_data.columns = sp500_data.columns.str.capitalize()
sp500_data = sp500_data.dropna(subset=['Close'])

# Load Gig Economy Data
# gig_data = pd.read_csv('../../data/yfinance/gig_yfinance_monthly.csv')
gig_data = pd.read_csv('../../data/yfinance/gig_yfinance_monthly.csv')
gig_data_edgar = pd.read_csv('../../data/edgar_sec/gig_edgar_sec.csv')
gig_data.columns = gig_data.columns.str.capitalize()
gig_data = gig_data.dropna(subset=['Close'])

# Capitalize Date and Ticker columns ONLY
sp500_data_edgar.rename(columns={'date': 'Date', 'ticker': 'Ticker'}, inplace=True)
gig_data_edgar.rename(columns={'date': 'Date', 'ticker': 'Ticker'}, inplace=True)
sp500_data_edgar = sp500_data_edgar[['Date', 'Ticker', 'NetIncomeLoss', 'NetWorth', 'EarningsPerShareDiluted', 'SharesOutstanding']].copy()
gig_data_edgar = gig_data_edgar[['Date', 'Ticker', 'NetIncomeLoss', 'NetWorth', 'EarningsPerShareDiluted', 'SharesOutstanding']].copy()


# capitalize all columns
# merge gig_data with gig_data_edgar on Date and Ticker
gig_data = gig_data.loc[:, ~gig_data.columns.duplicated()]
sp500_data = sp500_data.loc[:, ~sp500_data.columns.duplicated()]
gig_merged = pd.merge(gig_data, gig_data_edgar, on=['Date', 'Ticker'], how='left', suffixes=('', '_edgar'))
sp500_merged = pd.merge(sp500_data, sp500_data_edgar, on=['Date', 'Ticker'], how='left', suffixes=('', '_edgar'))

edgar_cols = ['NetIncomeLoss', 'NetWorth', 'EarningsPerShareDiluted', 'SharesOutstanding']
gig_merged.sort_values(by=['Ticker', 'Date'], inplace=True)
gig_merged[edgar_cols] = gig_merged.groupby('Ticker')[edgar_cols].transform(lambda x: x.bfill().ffill())
gig_merged

sp500_merged.sort_values(by=['Ticker', 'Date'], inplace=True)
sp500_merged[edgar_cols] = sp500_merged.groupby('Ticker')[edgar_cols].transform(lambda x: x.bfill().ffill())
sp500_merged

sp500_data = sp500_merged.copy()
gig_data = gig_merged.copy()

# Display shapes and date ranges
def data_summary(df, name):
    return {
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Date Min': df['Date'].min(),
        'Date Max': df['Date'].max(),
        'Companies': df['Ticker'].nunique(),
        'Missing Values': df.isnull().sum().sum()
    }

summary_df = pd.DataFrame([
    data_summary(sp500_data, 'S&P 500'),
    data_summary(gig_data, 'Gig Economy')
])

# Altair summary table
alt.Chart(summary_df).mark_text().encode(
    y=alt.Y('Dataset:N', axis=alt.Axis(title=None)),
    text='Rows:Q',
    color=alt.condition(
        alt.datum.Rows > 0,
        alt.value('black'),
        alt.value('red')
    )
).properties(title='Dataset Summary (Rows)')

summary_df

,Dataset,Rows,Columns,Date Min,Date Max,Companies,Missing Values
0,S&P 500,95684,14,2009-01-31,2025-12-31,501,1844
1,Gig Economy,1531,14,2009-01-31,2025-12-31,13,0


In [3]:
# Add category labels
sp500_data['Category'] = 'sp500'
gig_data['Category'] = 'gig-work'

# if Ticker exists in both datsets, removee the duplicate from S&P 500 data
overlap_tickers = set(sp500_data['Ticker']).intersection(set(gig_data['Ticker']))
print(f"Overlapping Tickers: {overlap_tickers}")
sp500_data = sp500_data[~sp500_data['Ticker'].isin(overlap_tickers)]

# Combine datasets
combined_data = pd.concat([sp500_data, gig_data], ignore_index=True)

combined_data['MarketCap'] = combined_data['Close'] * combined_data['SharesOutstanding']

combined_data = combined_data.drop(columns=[
    # 'Industrykey', 
    'Sectorkey',
], errors='ignore')

input_data = combined_data.dropna().copy()

# Altair: Show company count by category interactively
cat_count = input_data.groupby('Category').size().reset_index(name='Count')
alt.Chart(cat_count).mark_bar().encode(
    x=alt.X('Category:N', sort='-y', title='Category'),
    y=alt.Y('Count:Q', title='Number of Companies'),
    tooltip=['Category', 'Count']
).properties(title='Company Count by Category').interactive()

input_data.head()

Overlapping Tickers: {'DASH', 'UBER'}


,Date,Month,Quarter,Year,Company,Ticker,Industrykey,Close,Volume_accum,NetIncomeLoss,NetWorth,EarningsPerShareDiluted,SharesOutstanding,Category,MarketCap
0,2009-01-31,1,1,2009,Agilent Technologies,A,diagnostics-research,11.443532,143548458.0,4.288339e+08,2.559000e+09,1.57,394.0,sp500,4508.751604
1,2009-02-28,2,1,2009,Agilent Technologies,A,diagnostics-research,8.778862,119895557.0,4.288339e+08,2.559000e+09,1.57,394.0,sp500,3458.871628
2,2009-03-31,3,1,2009,Agilent Technologies,A,diagnostics-research,9.728273,122505060.0,4.288339e+08,2.559000e+09,1.57,394.0,sp500,3832.939716
3,2009-04-30,4,2,2009,Agilent Technologies,A,diagnostics-research,11.557462,101640751.0,4.288339e+08,2.559000e+09,1.57,394.0,sp500,4553.639925
4,2009-05-31,5,2,2009,Agilent Technologies,A,diagnostics-research,11.538473,79312735.0,4.288339e+08,2.559000e+09,1.57,394.0,sp500,4546.158413


In [4]:
# Group by Ticker and determine earliest data point(start date) for each company
earliest_start_dates = input_data.groupby('Ticker')['Date'].min().reset_index()
earliest_start_dates.columns = ['Ticker', 'EarliestStartDate']
earliest_start_dates = earliest_start_dates.sort_values(by='EarliestStartDate', ascending=True)

earliest_start_dates['EarliestStartDate'] = pd.to_datetime(earliest_start_dates['EarliestStartDate'])

# Use start date near 90.5th percentile of all earliest start dates as data range cutoff(captures approximately 5 years before and after COVID)
start_date_to_use = earliest_start_dates['EarliestStartDate'].iloc[int(len(earliest_start_dates) * 0.905)]  # 90.5th percentile
print(f"Start Date to Use: {start_date_to_use.date()}")
input_data_clean = input_data[input_data['Date'] >= str(start_date_to_use)].copy()
print(f"\nNumber of Companies after cleaning: {input_data_clean['Ticker'].nunique()}")

# 'HLF', 'NUS',  'PRI', 'USNA' to mlm-work Category
mlm_tickers = ['HLF', 'NUS', 'PRI', 'USNA']
input_data_clean.loc[input_data_clean['Ticker'].isin(mlm_tickers), 'Category'] = 'mlm-work'
input_data_clean.reset_index(drop=True, inplace=True)

gig_tickers = input_data_clean[input_data_clean['Category'] == 'gig-work']['Ticker'].unique()
print(f"Gig Economy Tickers in Cleaned Data: {gig_tickers}")
mlm_tickers_in_data = input_data_clean[input_data_clean['Category'] == 'mlm-work']['Ticker'].unique()
print(f"MLM Tickers in Cleaned Data: {mlm_tickers_in_data}")

# Remove known outliers (AMCR and FANG) based on visualizations and domain knowledge
# input_data_clean = input_data_clean[~input_data_clean['Ticker'].isin(['AMCR', 'FANG'])].copy()
input_data_clean.head()

Start Date to Use: 2014-10-31

Number of Companies after cleaning: 506
Gig Economy Tickers in Cleaned Data: <ArrowStringArray>
['CART', 'DASH', 'ETSY', 'FVRR', 'LYFT', 'SHOP', 'UBER', 'UDMY', 'UPWK']
Length: 9, dtype: str
MLM Tickers in Cleaned Data: <ArrowStringArray>
['HLF', 'NUS', 'PRI', 'USNA']
Length: 4, dtype: str


,Date,Month,Quarter,Year,Company,Ticker,Industrykey,Close,Volume_accum,NetIncomeLoss,NetWorth,EarningsPerShareDiluted,SharesOutstanding,Category,MarketCap
0,2014-11-30,11,4,2014,Agilent Technologies,A,diagnostics-research,38.862274,56373700.0,-7.982861e+08,5.289000e+09,3.27,3.480000e+08,sp500,1.352407e+10
1,2014-12-31,12,4,2014,Agilent Technologies,A,diagnostics-research,37.225567,48770200.0,-7.982861e+08,5.289000e+09,3.27,3.480000e+08,sp500,1.295450e+10
2,2015-01-31,1,1,2015,Agilent Technologies,A,diagnostics-research,34.427284,53086600.0,2.036497e+09,5.301000e+09,0.58,1.332000e+09,sp500,4.585714e+10
3,2015-02-28,2,1,2015,Agilent Technologies,A,diagnostics-research,38.474335,52418300.0,2.036497e+09,5.301000e+09,0.58,1.332000e+09,sp500,5.124781e+10
4,2015-03-31,3,1,2015,Agilent Technologies,A,diagnostics-research,37.966026,44856400.0,2.036497e+09,5.301000e+09,0.58,1.332000e+09,sp500,5.057075e+10


In [5]:
# condense to quartely data prior to calculating changes and rolling stats since financials are quarterly()
input_data_clean['date'] = pd.to_datetime(input_data_clean['Date'])
input_data_clean['ticker'] = input_data_clean['Ticker']
input_data_clean = input_data_clean.sort_values(by=['ticker', 'date'])
input_data_clean = input_data_clean.groupby(['ticker']).resample('QE', on='date').last().reset_index(drop=True)

input_data_clean.isna().sum()

Date                       0
Month                      0
Quarter                    0
Year                       0
Company                    0
Ticker                     0
Industrykey                0
Close                      0
Volume_accum               0
NetIncomeLoss              0
NetWorth                   0
EarningsPerShareDiluted    0
SharesOutstanding          0
Category                   0
MarketCap                  0
dtype: int64

In [6]:
input_data_clean

,Date,Month,Quarter,Year,Company,Ticker,Industrykey,Close,Volume_accum,NetIncomeLoss,NetWorth,EarningsPerShareDiluted,SharesOutstanding,Category,MarketCap
0,2014-12-31,12,4,2014,Agilent Technologies,A,diagnostics-research,37.225567,48770200.0,-7.982861e+08,5.289000e+09,3.27,3.480000e+08,sp500,1.295450e+10
1,2015-03-31,3,1,2015,Agilent Technologies,A,diagnostics-research,37.966026,44856400.0,2.036497e+09,5.301000e+09,0.58,1.332000e+09,sp500,5.057075e+10
2,2015-06-30,6,2,2015,Agilent Technologies,A,diagnostics-research,35.340466,58978100.0,-1.279037e+09,5.301000e+09,0.99,6.660000e+08,sp500,2.353675e+10
3,2015-09-30,9,3,2015,Agilent Technologies,A,diagnostics-research,31.538609,47113000.0,3.854725e+09,5.301000e+09,1.42,4.440000e+08,sp500,1.400314e+10
4,2015-12-31,12,4,2015,Agilent Technologies,A,diagnostics-research,38.515438,59284400.0,3.894000e+09,5.304000e+09,2.13,3.410000e+08,sp500,1.313376e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21888,2024-12-31,12,4,2024,Zoetis,ZTS,drug-manufacturers-specialty-generic,160.109528,50091600.0,2.114000e+09,4.991000e+09,4.49,4.688910e+08,sp500,7.507392e+10
21889,2025-03-31,3,1,2025,Zoetis,ZTS,drug-manufacturers-specialty-generic,162.288910,52813200.0,5.990000e+08,4.770000e+09,1.31,1.832000e+09,sp500,2.973133e+11
21890,2025-06-30,6,2,2025,Zoetis,ZTS,drug-manufacturers-specialty-generic,154.232376,58660600.0,1.223000e+09,4.770000e+09,2.67,9.134000e+08,sp500,1.408759e+11
21891,2025-09-30,9,3,2025,Zoetis,ZTS,drug-manufacturers-specialty-generic,145.192337,58980800.0,1.905000e+09,4.770000e+09,4.18,6.072000e+08,sp500,8.816079e+10


In [7]:
# Convert Date to datetime
input_data_clean['Date'] = pd.to_datetime(input_data_clean['Date'])

# Calculate returns and log returns
input_data_clean = input_data_clean.sort_values(['Ticker', 'Date'])
input_data_clean['Return'] = input_data_clean.groupby('Ticker')['Close'].pct_change()
input_data_clean['Log_Return'] = input_data_clean.groupby('Ticker')['Close'].transform(lambda x: np.log(x / x.shift(1)))

# Volume changes
input_data_clean['Volume_Change'] = input_data_clean.groupby('Ticker')['Volume_accum'].pct_change()

# NetIncomeLoss, NetWorthm and EarningsPerShareDiluted (3-month pct changes since financials are quarterly)
input_data_clean['NIL_Change'] = input_data_clean.groupby('Ticker')['NetIncomeLoss'].pct_change()
input_data_clean['NetWorth_Change'] = input_data_clean.groupby('Ticker')['NetWorth'].pct_change()
input_data_clean['EPS_Change'] = input_data_clean.groupby('Ticker')['EarningsPerShareDiluted'].pct_change()

# Rolling statistics (2 & 4-quarter window)
for ticker in input_data_clean['Ticker'].unique():
    mask = input_data_clean['Ticker'] == ticker
    input_data_clean.loc[mask, 'Volatility_2Q'] = input_data_clean.loc[mask, 'Return'].rolling(window=2).std()
    input_data_clean.loc[mask, 'Avg_Return_2Q'] = input_data_clean.loc[mask, 'Return'].rolling(window=2).mean()
    input_data_clean.loc[mask, 'Avg_Volume_Change_2Q'] = input_data_clean.loc[mask, 'Volume_Change'].rolling(window=2).mean()
    input_data_clean.loc[mask, 'NIL_Volatility_2Q'] = input_data_clean.loc[mask, 'NIL_Change'].rolling(window=2).std()
    input_data_clean.loc[mask, 'Avg_NIL_Change_2Q'] = input_data_clean.loc[mask, 'NIL_Change'].rolling(window=2).mean()
    input_data_clean.loc[mask, 'NetWorth_Volatility_2Q'] = input_data_clean.loc[mask, 'NetWorth_Change'].rolling(window=2).std()
    input_data_clean.loc[mask, 'Avg_NetWorth_Change_2Q'] = input_data_clean.loc[mask, 'NetWorth_Change'].rolling(window=2).mean()
    input_data_clean.loc[mask, 'EPS_Volatility_2Q'] = input_data_clean.loc[mask, 'EPS_Change'].rolling(window=2).std()
    input_data_clean.loc[mask, 'Avg_EPS_Change_2Q'] = input_data_clean.loc[mask, 'EPS_Change'].rolling(window=2).mean()
    input_data_clean.loc[mask, 'Volatility_4Q'] = input_data_clean.loc[mask, 'Return'].rolling(window=4).std()
    input_data_clean.loc[mask, 'Avg_Return_4Q'] = input_data_clean.loc[mask, 'Return'].rolling(window=4).mean()
    input_data_clean.loc[mask, 'Avg_Volume_Change_4Q'] = input_data_clean.loc[mask, 'Volume_Change'].rolling(window=4).mean()
    input_data_clean.loc[mask, 'NIL_Volatility_4Q'] = input_data_clean.loc[mask, 'NIL_Change'].rolling(window=4).std()
    input_data_clean.loc[mask, 'Avg_NIL_Change_4Q'] = input_data_clean.loc[mask, 'NIL_Change'].rolling(window=4).mean()
    input_data_clean.loc[mask, 'NetWorth_Volatility_4Q'] = input_data_clean.loc[mask, 'NetWorth_Change'].rolling(window=4).std()
    input_data_clean.loc[mask, 'Avg_NetWorth_Change_4Q'] = input_data_clean.loc[mask, 'NetWorth_Change'].rolling(window=4).mean()
    input_data_clean.loc[mask, 'EPS_Volatility_4Q'] = input_data_clean.loc[mask, 'EPS_Change'].rolling(window=4).std()
    input_data_clean.loc[mask, 'Avg_EPS_Change_4Q'] = input_data_clean.loc[mask, 'EPS_Change'].rolling(window=4).mean()

input_df = input_data_clean.dropna()

input_df.head()

,Date,Month,Quarter,Year,Company,Ticker,Industrykey,Close,Volume_accum,NetIncomeLoss,...,Avg_EPS_Change_2Q,Volatility_4Q,Avg_Return_4Q,Avg_Volume_Change_4Q,NIL_Volatility_4Q,Avg_NIL_Change_4Q,NetWorth_Volatility_4Q,Avg_NetWorth_Change_4Q,EPS_Volatility_4Q,Avg_EPS_Change_4Q
4,2015-12-31,12,4,2015,Agilent Technologies,A,diagnostics-research,38.515438,59284400.0,3.894000e+09,...,0.467172,0.146801,0.016093,0.072934,1.852083,-2.295682,0.001074,0.000709,0.694633,0.204653
5,2016-03-31,3,1,2016,Agilent Technologies,A,diagnostics-research,36.709873,37122800.0,1.812613e+09,...,-0.205399,0.149987,-0.000599,-0.000458,1.783385,-1.541538,0.106995,-0.053309,0.738134,0.182610
6,2016-06-30,6,2,2016,Agilent Technologies,A,diagnostics-research,41.090767,52011000.0,5.812048e+08,...,0.228811,0.150876,0.046524,0.021100,1.830505,-1.304362,0.106995,-0.053309,0.940991,0.347992
7,2016-09-30,9,3,2016,Agilent Technologies,A,diagnostics-research,43.727905,38864400.0,2.502968e+09,...,1.050877,0.111773,0.089463,0.008203,1.877489,0.525710,0.106995,-0.053309,0.961782,0.422739
8,2016-12-31,12,4,2016,Agilent Technologies,A,diagnostics-research,42.429649,34086300.0,4.048000e+09,...,0.905128,0.078698,0.026737,-0.087119,1.846201,0.677482,0.106900,-0.053450,1.018799,0.566970


In [8]:
# distribution to tickers per category
cat_counts = input_df.groupby('Category')['Ticker'].nunique().reset_index(name='CompanyCount')
for cat in cat_counts['Category']:
    count = cat_counts[cat_counts['Category'] == cat]['CompanyCount'].values[0]
    print(f"{cat}: {count} companies")
    
average_companies_per_category = cat_counts['CompanyCount'].mean()
print(f"\nAverage number of companies per category: {average_companies_per_category:.2f}")

gig-work: 9 companies
mlm-work: 4 companies
sp500: 490 companies

Average number of companies per category: 167.67


In [9]:
# Aggregate metrics by ticker for clustering
ticker_features = input_df.groupby(['Ticker', 'Company', 'Industrykey', 'Category']).agg({
    # 'Close': ['mean', 'std', 'min', 'max'],
    'Return': ['mean', 'std', 'skew'],
    'Volume_accum': ['mean', 'std', 'max'],
    # 'SharesOutstanding': ['mean', 'std'],
    'MarketCap': ['mean', 'std', 'min', 'max'],
    'Volume_Change': ['mean', 'std'],
    'EPS_Change': ['mean', 'std'],
    'NetWorth_Change': ['mean', 'std'],
    'NIL_Change': ['mean', 'std'],
    # 'Log_Return': ['mean', 'std', 'skew'],
    'Volatility_2Q': ['mean', 'max'],
    'Avg_Return_2Q': 'mean',
    'Avg_Volume_Change_2Q': 'mean',
    'EPS_Volatility_2Q': ['mean', 'max'],
    'Avg_EPS_Change_2Q': 'mean',
    'NetWorth_Volatility_2Q': ['mean', 'max'],
    'Avg_NetWorth_Change_2Q': 'mean',
    'NIL_Volatility_2Q': ['mean', 'max'],
    'Avg_NIL_Change_2Q': 'mean',
    'Volatility_4Q': ['mean', 'max'],
    'Avg_Return_4Q': 'mean',
    'Avg_Volume_Change_4Q': 'mean',
    'EPS_Volatility_4Q': ['mean', 'max'],
    'Avg_EPS_Change_4Q': 'mean',
    'NetWorth_Volatility_4Q': ['mean', 'max'],
    'Avg_NetWorth_Change_4Q': 'mean',
    'NIL_Volatility_4Q': ['mean', 'max'],
    'Avg_NIL_Change_4Q': 'mean'
}).reset_index()

# Flatten column names
ticker_features.columns = ['_'.join(col).strip('_') for col in ticker_features.columns.values]
ticker_features.rename(columns={'Ticker_': 'Ticker', 'Company_': 'Company', 'Industrykey_': 'Industrykey', 'Category_': 'Category'}, inplace=True)
ticker_non_numeric = ['Ticker', 'Company', 'Industrykey', 'Category']

print(f"\nTicker-level features shape: {ticker_features.shape}")
print(f"\nFeatures:\n{ticker_features.columns.tolist()}")
ticker_features = ticker_features.dropna()
ticker_features.head()


Ticker-level features shape: (503, 48)

Features:
['Ticker', 'Company', 'Industrykey', 'Category', 'Return_mean', 'Return_std', 'Return_skew', 'Volume_accum_mean', 'Volume_accum_std', 'Volume_accum_max', 'MarketCap_mean', 'MarketCap_std', 'MarketCap_min', 'MarketCap_max', 'Volume_Change_mean', 'Volume_Change_std', 'EPS_Change_mean', 'EPS_Change_std', 'NetWorth_Change_mean', 'NetWorth_Change_std', 'NIL_Change_mean', 'NIL_Change_std', 'Volatility_2Q_mean', 'Volatility_2Q_max', 'Avg_Return_2Q_mean', 'Avg_Volume_Change_2Q_mean', 'EPS_Volatility_2Q_mean', 'EPS_Volatility_2Q_max', 'Avg_EPS_Change_2Q_mean', 'NetWorth_Volatility_2Q_mean', 'NetWorth_Volatility_2Q_max', 'Avg_NetWorth_Change_2Q_mean', 'NIL_Volatility_2Q_mean', 'NIL_Volatility_2Q_max', 'Avg_NIL_Change_2Q_mean', 'Volatility_4Q_mean', 'Volatility_4Q_max', 'Avg_Return_4Q_mean', 'Avg_Volume_Change_4Q_mean', 'EPS_Volatility_4Q_mean', 'EPS_Volatility_4Q_max', 'Avg_EPS_Change_4Q_mean', 'NetWorth_Volatility_4Q_mean', 'NetWorth_Volatility

,Ticker,Company,Industrykey,Category,Return_mean,Return_std,Return_skew,Volume_accum_mean,Volume_accum_std,Volume_accum_max,...,Avg_Volume_Change_4Q_mean,EPS_Volatility_4Q_mean,EPS_Volatility_4Q_max,Avg_EPS_Change_4Q_mean,NetWorth_Volatility_4Q_mean,NetWorth_Volatility_4Q_max,Avg_NetWorth_Change_4Q_mean,NIL_Volatility_4Q_mean,NIL_Volatility_4Q_max,Avg_NIL_Change_4Q_mean
0,A,Agilent Technologies,diagnostics-research,sp500,0.043067,0.117935,-0.036899,4.063654e+07,9.984756e+06,7.083990e+07,...,0.049141,0.981866,3.123564,0.273767,0.037271,0.106995,0.003843,3.953183,18.616855,1.651669
1,AAPL,Apple Inc.,consumer-electronics,sp500,0.071759,0.156266,-0.070810,2.275765e+09,1.038062e+09,6.280072e+09,...,0.014215,0.542979,0.635845,0.133357,0.067775,0.138963,-0.013587,0.544057,0.631385,0.157407
2,ABBV,AbbVie,drug-manufacturers-general,sp500,0.053754,0.125594,0.308907,1.482559e+08,4.964051e+07,3.681825e+08,...,0.059068,1.042537,4.408229,0.440800,0.132883,0.464296,0.052540,1.126112,4.297516,0.460053
3,ABNB,Airbnb,travel-services,sp500,0.008785,0.198991,-0.249100,1.157236e+08,3.252025e+07,2.084769e+08,...,0.012426,7.451058,11.924797,3.378764,0.154001,0.322918,0.076989,5.874528,9.007148,2.637675
4,ABT,Abbott Laboratories,medical-devices,sp500,0.037560,0.097236,-0.021515,1.296637e+08,4.166209e+07,3.113584e+08,...,0.072598,1.016708,3.318321,0.444757,0.111448,0.497696,0.024302,1.018145,3.335739,0.441611


In [10]:
# Select numeric features for clustering
feature_cols = [col for col in ticker_features.columns if col not in ticker_non_numeric]
X = ticker_features[feature_cols].values
company_labels = ticker_features['Ticker'].values

X_scaled = robust_scaler.fit_transform(X)

In [11]:
gig_work_stats = ticker_features[ticker_features['Category'] == 'gig-work'][feature_cols].mean()
mlm_work_stats = ticker_features[ticker_features['Category'] == 'mlm-work'][feature_cols].mean()
sp500_stats = ticker_features[(ticker_features['Category'] == 'sp500')][feature_cols].mean()
print("\nGig Work Feature Statistics:")
print(gig_work_stats)
print("\nMLM Work Feature Statistics:")
print(mlm_work_stats)
print("\nS&P 500 Categories Feature Statistics (Mean):")
print(sp500_stats)


Gig Work Feature Statistics:
Return_mean                    6.010165e-02
Return_std                     3.126608e-01
Return_skew                    8.148795e-01
Volume_accum_mean              1.576998e+08
Volume_accum_std               6.718336e+07
Volume_accum_max               3.359505e+08
MarketCap_mean                 5.032708e+10
MarketCap_std                  4.571441e+10
MarketCap_min                  4.624906e+09
MarketCap_max                  1.823447e+11
Volume_Change_mean             1.083943e-01
Volume_Change_std              5.066699e-01
EPS_Change_mean                6.241443e-02
EPS_Change_std                 4.049834e+00
NetWorth_Change_mean           1.427594e-02
NetWorth_Change_std            1.702095e-01
NIL_Change_mean               -2.524363e-01
NIL_Change_std                 3.218192e+00
Volatility_2Q_mean             2.147945e-01
Volatility_2Q_max              7.942174e-01
Avg_Return_2Q_mean             5.775776e-02
Avg_Volume_Change_2Q_mean      1.156282e-01
EP

In [12]:
# Correlation-Based Feature Clustering

# Compute Spearman correlation matrix (rank-based, robust to outliers)
corr_matrix, _ = spearmanr(ticker_features[feature_cols])
corr_df = pd.DataFrame(corr_matrix, index=feature_cols, columns=feature_cols)

# Convert correlation to distance: d = 1 - |r|
# Features with r=1.0 have distance 0, r=0 have distance 1
distance_matrix = 1 - np.abs(corr_matrix)

# Convert square distance matrix to condensed form for linkage
condensed_dist = squareform(distance_matrix, checks=False)

# Agglomerative clustering (average linkage)
linkage_matrix = linkage(condensed_dist, method='average')

# Cut dendrogram at distance threshold = 0.10  (equiv. to |correlation| >= 0.90)
distance_threshold = 0.10
feature_clusters = fcluster(linkage_matrix, t=distance_threshold, criterion='distance')

# Build feature cluster assignments
feature_cluster_df = pd.DataFrame({
    'Feature': feature_cols,
    'Cluster': feature_clusters,
})

# Select one representative per cluster
# Strategy: pick the feature with highest variance (most information)
selected_features = []
cluster_info = []

for cluster_id in sorted(feature_cluster_df['Cluster'].unique()):
    cluster_members = feature_cluster_df[feature_cluster_df['Cluster'] == cluster_id]['Feature'].tolist()
    
    # Compute variance for each feature in this cluster
    variances = ticker_features[cluster_members].var()
    best_feature = variances.idxmax()
    
    selected_features.append(best_feature)
    cluster_info.append({
        'Cluster': cluster_id,
        'Size': len(cluster_members),
        'Representative': best_feature,
        'Members': ', '.join(cluster_members),
    })

cluster_summary_df = pd.DataFrame(cluster_info)

print(f"- Feature Reduction Summary -")
print(f"Original features:       {len(feature_cols)}")
print(f"Feature clusters found:  {len(selected_features)}")
print(f"Reduction:               {len(feature_cols) - len(selected_features)} features removed ({(1 - len(selected_features)/len(feature_cols))*100:.1f}%)")
print(f"\nClusters with multiple features (showing redundancy):")
print(cluster_summary_df[cluster_summary_df['Size'] > 1][['Cluster', 'Size', 'Representative']].to_string(index=False))

# Show a few example clusters in detail
print(f"Example Redundant Feature Clusters:")
for idx, row in cluster_summary_df[cluster_summary_df['Size'] > 1].head(5).iterrows():
    print(f"\nCluster {row['Cluster']} ({row['Size']} features) → Representative: {row['Representative']}")
    members = row['Members'].split(', ')
    for m in members[:5]:  # show first 5
        corr_to_rep = corr_df.loc[row['Representative'], m]
        print(f"  • {m:50s}  (r = {corr_to_rep:+.3f})")
    if len(members) > 5:
        print(f"  ... and {len(members)-5} more")

cluster_summary_df

- Feature Reduction Summary -
Original features:       44
Feature clusters found:  15
Reduction:               29 features removed (65.9%)

Clusters with multiple features (showing redundancy):
 Cluster  Size             Representative
       1     3     Avg_EPS_Change_4Q_mean
       2     3            NIL_Change_mean
       3     3       NetWorth_Change_mean
       4     5 NetWorth_Volatility_2Q_max
       5     3           Volume_accum_max
       6     3              MarketCap_max
       8     3                Return_mean
       9     3                 Return_std
      10     2          Volatility_2Q_max
      12     3         Volume_Change_mean
      14     5      NIL_Volatility_2Q_max
      15     5      EPS_Volatility_2Q_max
Example Redundant Feature Clusters:

Cluster 1 (3 features) → Representative: Avg_EPS_Change_4Q_mean
  • EPS_Change_mean                                     (r = +0.967)
  • Avg_EPS_Change_2Q_mean                              (r = +0.987)
  • Avg_EPS_Change_4Q

,Cluster,Size,Representative,Members
0,1,3,Avg_EPS_Change_4Q_mean,"EPS_Change_mean, Avg_EPS_Change_2Q_mean, Avg_E..."
1,2,3,NIL_Change_mean,"NIL_Change_mean, Avg_NIL_Change_2Q_mean, Avg_N..."
2,3,3,NetWorth_Change_mean,"NetWorth_Change_mean, Avg_NetWorth_Change_2Q_m..."
3,4,5,NetWorth_Volatility_2Q_max,"NetWorth_Change_std, NetWorth_Volatility_2Q_me..."
4,5,3,Volume_accum_max,"Volume_accum_mean, Volume_accum_std, Volume_ac..."
5,6,3,MarketCap_max,"MarketCap_mean, MarketCap_std, MarketCap_max"
6,7,1,MarketCap_min,MarketCap_min
7,8,3,Return_mean,"Return_mean, Avg_Return_2Q_mean, Avg_Return_4Q..."
8,9,3,Return_std,"Return_std, Volatility_2Q_mean, Volatility_4Q_..."
9,10,2,Volatility_2Q_max,"Volatility_2Q_max, Volatility_4Q_max"


In [13]:
# Visualization: Feature Dendrogram 
# Convert scipy linkage to plottable dendrogram data
dend = dendrogram(linkage_matrix, labels=feature_cols, no_plot=True)

# Extract dendrogram segments for Altair
segments = []
for i, (icoord, dcoord) in enumerate(zip(dend['icoord'], dend['dcoord'])):
    for j in range(len(icoord)-1):
        segments.append({
            'x0': icoord[j], 'y0': dcoord[j],
            'x1': icoord[j+1], 'y1': dcoord[j+1],
            'segment_id': i
        })

seg_df = pd.DataFrame(segments)

# Threshold line data
threshold_df = pd.DataFrame({
    'y': [distance_threshold],
    'label': [f'Threshold = {distance_threshold} (r ≥ 0.90)']
})

# Dendrogram plot
dendro_base = alt.Chart(seg_df).mark_line(color='steelblue', strokeWidth=1.2)
dendro_lines = dendro_base.encode(
    x=alt.X('x0:Q', title='Feature Index'),
    x2='x1:Q',
    y=alt.Y('y0:Q', title='Distance (1 - |Spearman r|)', scale=alt.Scale(zero=True)),
    y2='y1:Q',
    detail='segment_id:N'
)

threshold_line = alt.Chart(threshold_df).mark_rule(
    color='#E45756', strokeDash=[6, 4], strokeWidth=2
).encode(
    y='y:Q',
    tooltip=[alt.Tooltip('label:N', title='Cut Threshold')]
)

threshold_label = alt.Chart(threshold_df).mark_text(
    align='left', dx=5, dy=-8, fontSize=11, fontWeight='bold', color='#E45756'
).encode(
    x=alt.value(0),
    y='y:Q',
    text='label:N'
)

dendro_chart = (dendro_lines + threshold_line + threshold_label).properties(
    title='Feature Dendrogram — Correlation-Based Hierarchical Clustering',
    width=700,
    height=350
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_title(
    fontSize=16
)

dendro_chart

alt.LayerChart(...)

In [14]:
# Correlation Heatmap: After Reduction
# Show correlation structure for selected features only

selected_corr = corr_df.loc[selected_features, selected_features]

# Prepare data for Altair heatmap
corr_long = []
for i, feat1 in enumerate(selected_features):
    for j, feat2 in enumerate(selected_features):
        corr_long.append({
            'Feature1': feat1,
            'Feature2': feat2,
            'Correlation': selected_corr.loc[feat1, feat2],
            'AbsCorr': abs(selected_corr.loc[feat1, feat2])
        })

corr_long_df = pd.DataFrame(corr_long)

# Heatmap
heatmap = alt.Chart(corr_long_df).mark_rect().encode(
    x=alt.X('Feature1:N', title='', axis=alt.Axis(labelAngle=-45, labelLimit=200)),
    y=alt.Y('Feature2:N', title='', axis=alt.Axis(labelLimit=200)),
    color=alt.Color('Correlation:Q', 
                    scale=alt.Scale(scheme='redblue', domain=[-1, 1], reverse=True),
                    title='Spearman r'),
    tooltip=[
        alt.Tooltip('Feature1:N', title='Feature 1'),
        alt.Tooltip('Feature2:N', title='Feature 2'),
        alt.Tooltip('Correlation:Q', format='.3f', title='Correlation')
    ]
).properties(
    title=f'Correlation Heatmap — {len(selected_features)} Selected Features (After Reduction)',
    width=600,
    height=600
).configure_axis(
    labelFontSize=9,
    titleFontSize=12
).configure_title(
    fontSize=15
)

heatmap

alt.Chart(...)

In [15]:
# Create Reduced Feature Set
# Build reduced feature matrix
X_reduced = ticker_features[selected_features].values
X_reduced_scaled = robust_scaler.fit_transform(X_reduced)

# Compare PCA variance explained: before vs after
pca_original = PCA(random_state=42).fit(X_scaled)
pca_reduced = PCA(random_state=42).fit(X_reduced_scaled)

cumvar_original = np.cumsum(pca_original.explained_variance_ratio_)
cumvar_reduced = np.cumsum(pca_reduced.explained_variance_ratio_)

print("- PCA Variance Comparison: Original vs Reduced Features -")
print(f"{'Metric':<40} {'Original (44)':<18} {'Reduced ({})'.format(len(selected_features)):<18}")

for n_comp in [2, 3, 5, 10]:
    if n_comp <= len(selected_features):
        orig_var = cumvar_original[n_comp-1] if n_comp <= len(cumvar_original) else cumvar_original[-1]
        red_var = cumvar_reduced[n_comp-1] if n_comp <= len(cumvar_reduced) else cumvar_reduced[-1]
        print(f"{n_comp} components: {orig_var:>16.1%} {red_var:>16.1%}")

n_95_orig = np.argmax(cumvar_original >= 0.95) + 1 if any(cumvar_original >= 0.95) else len(cumvar_original)
n_95_red = np.argmax(cumvar_reduced >= 0.95) + 1 if any(cumvar_reduced >= 0.95) else len(cumvar_reduced)
print(f"{'Components for 95% var:':<40} {n_95_orig:>16} {n_95_red:>16}")

n_99_orig = np.argmax(cumvar_original >= 0.99) + 1 if any(cumvar_original >= 0.99) else len(cumvar_original)
n_99_red = np.argmax(cumvar_reduced >= 0.99) + 1 if any(cumvar_reduced >= 0.99) else len(cumvar_reduced)
print(f"{'Components for 99% var:':<40} {n_99_orig:>16} {n_99_red:>16}")

# Check max pairwise correlation in reduced set
max_corr_reduced = selected_corr.where(~np.eye(len(selected_features), dtype=bool)).abs().max().max()
print(f"\n{'Max |correlation| in reduced set:':<40} {max_corr_reduced:>16.3f}")

print("Feature set successfully reduced while preserving variance structure")
print(f"  New feature count: {len(selected_features)}")
print(f"  Features removed: {len(feature_cols) - len(selected_features)}")
print(f"  Max correlation reduced from >0.99 to {max_corr_reduced:.3f}")

- PCA Variance Comparison: Original vs Reduced Features -
Metric                                   Original (44)      Reduced (15)      
2 components:           100.0%           100.0%
3 components:           100.0%           100.0%
5 components:           100.0%           100.0%
10 components:           100.0%           100.0%
Components for 95% var:                                 2                2
Components for 99% var:                                 2                2

Max |correlation| in reduced set:                   0.869
Feature set successfully reduced while preserving variance structure
  New feature count: 15
  Features removed: 29
  Max correlation reduced from >0.99 to 0.869


In [16]:
# Update Global Feature Set to Use Reduced Features
# This ensures all downstream clustering (DBSCAN, K-Means) uses the reduced set

# Store original for reference
feature_cols_original = feature_cols.copy()
X_scaled_original = X_scaled.copy()

# Replace with reduced features
feature_cols = selected_features
X = X_reduced
X_scaled = X_reduced_scaled

print(f"Global feature_cols updated: {len(feature_cols_original)} to {len(feature_cols)} features")
print(f"X_scaled shape: {X_scaled_original.shape} to {X_scaled.shape}")

Global feature_cols updated: 44 to 15 features
X_scaled shape: (503, 44) to (503, 15)


In [17]:
## DBSCAN Clustering
# ── K-Distance Graph (Altair) ──────────────────────────────────────────────
k = 3
nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances_sorted = np.sort(distances[:, k - 1], axis=0)

kdist_df = pd.DataFrame({
    'Index': np.arange(len(distances_sorted)),
    'Distance': distances_sorted
})

# eps is at elbow point of k-distance graph, which is where the distance starts to increase sharply
eps_value = distances_sorted[int(len(distances_sorted) * 0.975)]  # heuristic: 97th percentile distance
print(f"Optimal eps value based on k-distance graph: {eps_value:.4f}")

eps_rule_df = pd.DataFrame({'eps': [float(eps_value)]})

line = alt.Chart(kdist_df).mark_line(color='steelblue').encode(
    x=alt.X('Index:Q', title='Data Points Sorted by Distance'),
    y=alt.Y('Distance:Q', title=f'{k}-Nearest Neighbor Distance', scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip('Index:Q'), alt.Tooltip('Distance:Q', format='.4f')]
)

eps_line = alt.Chart(eps_rule_df).mark_rule(
    color='#E45756', strokeDash=[6, 4], strokeWidth=1.8
).encode(
    y=alt.Y('eps:Q'),
    tooltip=[alt.Tooltip('eps:Q', title='eps', format='.4f')]
)

eps_label = alt.Chart(eps_rule_df).mark_text(
    align='left', dx=6, dy=-8, fontSize=12, fontWeight='bold', color='#E45756'
).encode(
    x=alt.value(0),
    y=alt.Y('eps:Q'),
    text=alt.value(f'eps = {float(eps_value):.4f}')
)

(line + eps_line + eps_label).properties(
    title=f'K-Distance Graph for DBSCAN Parameter Selection (k={k})',
    width=600,
    height=300
).configure_axis(
    labelFontSize=14,
    titleFontSize=14
).configure_title(
    fontSize=18
)


Optimal eps value based on k-distance graph: 128.7730


alt.LayerChart(...)

In [18]:

min_samples = 2

dbscan = DBSCAN(eps=eps_value, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(X_scaled)

ticker_features['DBSCAN_Cluster'] = dbscan_labels

n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"DBSCAN Results  (eps={float(eps_value):.4f}, min_samples={min_samples})")
print(f"Number of clusters : {n_clusters_dbscan}")
print(f"Number of noise pts: {n_noise}")

if n_clusters_dbscan > 1:
    valid_mask = dbscan_labels != -1
    valid_labels = dbscan_labels[valid_mask]
    valid_data = X_scaled[valid_mask]
    if len(set(valid_labels)) > 1:
        dbscan_silhouette = silhouette_score(valid_data, valid_labels)
        print(f"Silhouette Score (excl. noise): {dbscan_silhouette:.3f}")
        
# list all tickers labeled as noise
noise_tickers = ticker_features[ticker_features['DBSCAN_Cluster'] == -1]['Ticker'].tolist()
print(f"Tickers labeled as noise by DBSCAN: {noise_tickers}")

DBSCAN Results  (eps=128.7730, min_samples=2)
Number of clusters : 1
Number of noise pts: 11
Tickers labeled as noise by DBSCAN: ['ALLE', 'AMCR', 'ARE', 'DOW', 'HAL', 'ICE', 'LIN', 'MDT', 'SPG', 'SW', 'VRT']


In [19]:
# Remove companies with very limited data and outliers in terms of financials
input_data_minus_noise = input_df[~input_df['Ticker'].isin(noise_tickers)].copy()
ticker_features_minus_noise = ticker_features[~ticker_features['Ticker'].isin(noise_tickers)].copy()

# Select numeric features for clustering
feature_cols = [col for col in ticker_features_minus_noise.columns if col not in ticker_non_numeric]
X_minus_noise = ticker_features_minus_noise[feature_cols].values
company_labels_minus_noise = ticker_features_minus_noise['Ticker'].values

X_scaled_minus_noise = robust_scaler.fit_transform(X_minus_noise)

In [20]:
input_data_minus_noise

,Date,Month,Quarter,Year,Company,Ticker,Industrykey,Close,Volume_accum,NetIncomeLoss,...,Avg_EPS_Change_2Q,Volatility_4Q,Avg_Return_4Q,Avg_Volume_Change_4Q,NIL_Volatility_4Q,Avg_NIL_Change_4Q,NetWorth_Volatility_4Q,Avg_NetWorth_Change_4Q,EPS_Volatility_4Q,Avg_EPS_Change_4Q
4,2015-12-31,12,4,2015,Agilent Technologies,A,diagnostics-research,38.515438,59284400.0,3.894000e+09,...,0.467172,0.146801,0.016093,0.072934,1.852083,-2.295682,0.001074,0.000709,0.694633,0.204653
5,2016-03-31,3,1,2016,Agilent Technologies,A,diagnostics-research,36.709873,37122800.0,1.812613e+09,...,-0.205399,0.149987,-0.000599,-0.000458,1.783385,-1.541538,0.106995,-0.053309,0.738134,0.182610
6,2016-06-30,6,2,2016,Agilent Technologies,A,diagnostics-research,41.090767,52011000.0,5.812048e+08,...,0.228811,0.150876,0.046524,0.021100,1.830505,-1.304362,0.106995,-0.053309,0.940991,0.347992
7,2016-09-30,9,3,2016,Agilent Technologies,A,diagnostics-research,43.727905,38864400.0,2.502968e+09,...,1.050877,0.111773,0.089463,0.008203,1.877489,0.525710,0.106995,-0.053309,0.961782,0.422739
8,2016-12-31,12,4,2016,Agilent Technologies,A,diagnostics-research,42.429649,34086300.0,4.048000e+09,...,0.905128,0.078698,0.026737,-0.087119,1.846201,0.677482,0.106900,-0.053450,1.018799,0.566970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21888,2024-12-31,12,4,2024,Zoetis,ZTS,drug-manufacturers-specialty-generic,160.109528,50091600.0,2.114000e+09,...,0.315565,0.140071,-0.036924,0.250946,0.806271,0.284017,0.066773,0.033386,0.805358,0.282076
21889,2025-03-31,3,1,2025,Zoetis,ZTS,drug-manufacturers-specialty-generic,162.288910,52813200.0,5.990000e+08,...,-0.282873,0.122041,0.001662,-0.060331,0.801101,0.287108,0.022140,-0.011070,0.799938,0.285344
21890,2025-06-30,6,2,2025,Zoetis,ZTS,drug-manufacturers-specialty-generic,154.232376,58660600.0,1.223000e+09,...,0.164964,0.122697,-0.017611,0.069072,0.736011,0.243647,0.022140,-0.011070,0.732020,0.240264
21891,2025-09-30,9,3,2025,Zoetis,ZTS,drug-manufacturers-specialty-generic,145.192337,58980800.0,1.905000e+09,...,0.801856,0.073664,-0.064689,0.118542,0.744561,0.261227,0.022140,-0.011070,0.741664,0.259491


In [ ]:
# input_data_minus_noise.to_csv('../../data/clustering/final_input_data.csv', index=False)

In [22]:
# ── K-Means Metrics ────────────────────────────────────────────────────────
k_range = range(2, 10)
inertias = []
silhouette_scores = []
davies_bouldin_scores = []
calinski_harabasz_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled_minus_noise)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled_minus_noise, labels))
    davies_bouldin_scores.append(davies_bouldin_score(X_scaled_minus_noise, labels))
    calinski_harabasz_scores.append(calinski_harabasz_score(X_scaled_minus_noise, labels))
    
# # Convert to arrays for use in cell 11
inertias_arr       = np.array(inertias)
silhouette_arr     = np.array(silhouette_scores)
db_arr             = np.array(davies_bouldin_scores)
ch_arr             = np.array(calinski_harabasz_scores)

# ── Build tidy DataFrames ─────────────────────────────────────────────────
k_list = list(k_range)

inertia_df = pd.DataFrame({'k': k_list, 'Inertia': inertias})

shared_metrics_df = pd.DataFrame(
    [{'k': k, 'Method': 'K-Means',      'Silhouette': s, 'Davies-Bouldin': d, 'Calinski-Harabasz': c}
     for k, s, d, c in zip(k_list, silhouette_scores, davies_bouldin_scores, calinski_harabasz_scores)]
)

# ── Shared color scale ────────────────────────────────────────────────────
method_color = alt.Color(
    'Method:N',
    scale=alt.Scale(domain=['K-Means'], range=['#4C78A8']),
    title='Method',  legend=None
)

base_km     = alt.Chart(inertia_df).encode(
    x=alt.X('k:O', title='Number of Clusters (k)', axis=alt.Axis(labelAngle=0))
)
base_shared = alt.Chart(shared_metrics_df).encode(
    x=alt.X('k:O', title='Number of Clusters (k / n)', axis=alt.Axis(labelAngle=0)),
    color=method_color
)

elbow = base_km.mark_line(point=True, color='#4C78A8').encode(
    y=alt.Y('Inertia:Q', title='Inertia', scale=alt.Scale(zero=False)),
    tooltip=['k', 'Inertia']
).properties(
    title='Elbow Method / Inertia  (K-Means Only)',
    width=alt.Step(60), height=300
)

silhouette_chart = base_shared.mark_line(point=True).encode(
    y=alt.Y('Silhouette:Q', title='Silhouette Score', scale=alt.Scale(zero=False)),
    tooltip=['k', 'Method', 'Silhouette']
).properties(
    title='Silhouette Score (Higher is Better)',
    width=alt.Step(60), height=300
)

davies_chart = base_shared.mark_line(point=True).encode(
    y=alt.Y('Davies-Bouldin:Q', title='Davies-Bouldin Index', scale=alt.Scale(zero=False)),
    tooltip=['k', 'Method', 'Davies-Bouldin']
).properties(
    title='Davies-Bouldin Index (Lower is Better)',
    width=alt.Step(60), height=300
)

calinski_chart = base_shared.mark_line(point=True).encode(
    y=alt.Y('Calinski-Harabasz:Q', title='Calinski-Harabasz Score', scale=alt.Scale(zero=False)),
    tooltip=['k', 'Method', 'Calinski-Harabasz']
).properties(
    title='Calinski-Harabasz Score (Higher is Better)',
    width=alt.Step(60), height=300
)

(
    (elbow | silhouette_chart) & (davies_chart | calinski_chart)
).resolve_scale(
    y='independent'
).configure_axis(
    labelFontSize=14,
    titleFontSize=14,
).configure_title(
    fontSize=18
)


alt.VConcatChart(...)

In [23]:
# Normalize metrics (min-max scaling)
def minmax(x):
    x = np.array(x)
    return (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x))

# K-Means combined score (4 metrics including inertia)
norm_inertia    = 1 - minmax(inertias_arr)   # lower is better → invert
norm_silhouette = minmax(silhouette_arr)
norm_db         = 1 - minmax(db_arr)          # lower is better → invert
norm_ch         = minmax(ch_arr)
combined_score  = (norm_inertia + norm_silhouette + norm_db + norm_ch) / 4

# Store full metrics table for reference
metrics_df = pd.DataFrame({
    'k': list(k_range),
    'Inertia': inertias,
    'Silhouette': silhouette_scores,
    'Davies-Bouldin': davies_bouldin_scores,
    'Calinski-Harabasz': calinski_harabasz_scores,
    'Combined_Score': combined_score
})

# Select optimal k for K-Means and apply
top_k_idx  = np.argsort(combined_score)[-4:][::-1]
top_k      = [list(k_range)[i] for i in top_k_idx]
optimal_k  = top_k[0]

kmeans_final  = KMeans(n_clusters=optimal_k, random_state=42, n_init=20)
kmeans_labels = kmeans_final.fit_predict(X_scaled_minus_noise)
ticker_features_minus_noise['KMeans_Cluster'] = kmeans_labels

print("K-Means Metrics Summary:")
print(metrics_df.to_string(index=False))
print(f"\nTop 4 k by combined normalized metrics (K-Means): {top_k}")
print(f"Optimal k selected for K-Means: {optimal_k}")

# ── Combined score comparison chart ──────────────────────────────────────
combined_scores_df = pd.DataFrame(
    [{'k': k, 'Method': 'K-Means', 'Combined_Score': s}
     for k, s in zip(list(k_range), combined_score)]
    
)

alt.Chart(combined_scores_df).mark_line(point=True).encode(
    x=alt.X('k:O', title='Number of Clusters (k / n)', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Combined_Score:Q', title='Combined Score (Normalized)', scale=alt.Scale(zero=False)),
    color=alt.Color(
        'Method:N',
        scale=alt.Scale(domain=['K-Means'], range=['#4C78A8']),
        title='Method'
    ),
    tooltip=['k', 'Method', alt.Tooltip('Combined_Score:Q', format='.4f')]
).properties(
    title='Combined Normalized Score — K-Means (Higher is Better)',
    width=alt.Step(60),
    height=320
).configure_axis(
    labelFontSize=14,
    titleFontSize=14,
).configure_title(
    fontSize=18
)


K-Means Metrics Summary:
 k       Inertia  Silhouette  Davies-Bouldin  Calinski-Harabasz  Combined_Score
 2 413531.122974    0.925595        0.227447         258.245711        0.750000
 3 343411.014224    0.837565        0.900013         205.094511        0.286526
 4 294584.768430    0.758618        0.756935         186.027463        0.233789
 5 244466.799858    0.767350        0.759259         192.738985        0.313797
 6 206943.865416    0.763680        0.823716         199.399261        0.344833
 7 190857.014393    0.706064        0.822259         186.614275        0.253248
 8 170036.039228    0.712859        0.630308         187.638032        0.355567
 9 152750.269888    0.698011        0.692223         189.217490        0.338281

Top 4 k by combined normalized metrics (K-Means): [2, 8, 6, 9]
Optimal k selected for K-Means: 2


alt.Chart(...)

In [ ]:
# save top_k as well as DBSCAN eps, and Z to csv
optimal_params_df = pd.DataFrame({
    'Parameter': ['DBSCAN_eps', 'DBSCAN_min_samples'],
    'Value': [float(eps_value), min_samples]
})
# optimal_params_df.to_csv('../../data/clustering/optimal_db_clustering_params.csv', index=False)

# save top_k to csv
top_params_df = pd.DataFrame({
    'Method': ['K-Means'] * len(top_k),
    'Top_Values': top_k
})
# top_params_df.to_csv('../../data/clustering/top_k_clustering_params.csv', index=False)

In [ ]:
# save ticker features to csv
# ticker_features_minus_noise.to_csv('../../data/clustering/ticker_features.csv', index=False)
ticker_features_minus_noise.head()

,Ticker,Company,Industrykey,Category,Return_mean,Return_std,Return_skew,Volume_accum_mean,Volume_accum_std,Volume_accum_max,...,EPS_Volatility_4Q_max,Avg_EPS_Change_4Q_mean,NetWorth_Volatility_4Q_mean,NetWorth_Volatility_4Q_max,Avg_NetWorth_Change_4Q_mean,NIL_Volatility_4Q_mean,NIL_Volatility_4Q_max,Avg_NIL_Change_4Q_mean,DBSCAN_Cluster,KMeans_Cluster
0,A,Agilent Technologies,diagnostics-research,sp500,0.043067,0.117935,-0.036899,4.063654e+07,9.984756e+06,7.083990e+07,...,3.123564,0.273767,0.037271,0.106995,0.003843,3.953183,18.616855,1.651669,0,0
1,AAPL,Apple Inc.,consumer-electronics,sp500,0.071759,0.156266,-0.070810,2.275765e+09,1.038062e+09,6.280072e+09,...,0.635845,0.133357,0.067775,0.138963,-0.013587,0.544057,0.631385,0.157407,0,0
2,ABBV,AbbVie,drug-manufacturers-general,sp500,0.053754,0.125594,0.308907,1.482559e+08,4.964051e+07,3.681825e+08,...,4.408229,0.440800,0.132883,0.464296,0.052540,1.126112,4.297516,0.460053,0,0
3,ABNB,Airbnb,travel-services,sp500,0.008785,0.198991,-0.249100,1.157236e+08,3.252025e+07,2.084769e+08,...,11.924797,3.378764,0.154001,0.322918,0.076989,5.874528,9.007148,2.637675,0,0
4,ABT,Abbott Laboratories,medical-devices,sp500,0.037560,0.097236,-0.021515,1.296637e+08,4.166209e+07,3.113584e+08,...,3.318321,0.444757,0.111448,0.497696,0.024302,1.018145,3.335739,0.441611,0,0
